# Day 47 · 1 — MySQL source and binlog CDC

**Problem:** Maintain current and historical product dimensions while resolving the latest
business state of each order, even when an older event arrives late.

Use a Python kernel with the required dependencies. All three notebooks must share the
same working directory and `DB`. Configure your own HDFS location in notebooks 2 and 3.
The database is isolated as `cdc_scd_db`; it represents the brief's ecommerce database.
No existing database is dropped. For a separate dataset, choose a different `DB` in all three notebooks.

**Execution order:** run setup, the initial data section and its capture cell, then notebooks 2 and 3.
For each subsequent change, capture the new events and rerun notebooks 2 and 3.
`Run All` applies every source change before downstream processing.
Reexecuting an already committed change or a no-change capture does not duplicate data.
Snowflake is deferred.

In [ ]:
import os
import re
from pathlib import Path

DB = "cdc_scd_db"  # Use the same database name in all three notebooks.
assert re.fullmatch(r"[a-zA-Z][a-zA-Z0-9_]{0,63}", DB)
LAB_DIR = Path.cwd() / "lab_data" / DB
LAB_DIR.mkdir(parents=True, exist_ok=True)
BRONZE = f"hdfs:///bronze/{DB}"
WAREHOUSE = f"hdfs:///warehouse/{DB}"
print("MySQL database:", DB)

## Install the MySQL dependencies

Run this cell once in the notebook kernel. `%pip` installs into the environment used by
this kernel. Pandas is already installed, so it is not included. The `rsa` extra supports
MySQL 8 password authentication; `mysql-replication` provides `pymysqlreplication`.
If Jupyter requests a kernel restart, restart it and rerun the configuration cell before
continuing. Skip this installation cell on subsequent executions when the packages are present.

In [ ]:
%pip install "PyMySQL[rsa]>=1.1,<2" "mysql-replication==1.0.17"

## Connect to MySQL

Lab credentials default to root/root; environment variables override them. We never print
the password. The kernel needs `PyMySQL` and `mysql-replication==1.0.17`.
This is real row-binlog CDC, not snapshot comparison.

Use an administrator account for the setup below. The reader needs REPLICATION SLAVE,
REPLICATION CLIENT and SELECT privileges; the lab's root account already has these.
The writer also needs permissions to create the lab database/tables and change their rows.

In [ ]:
import csv
import json
from datetime import datetime, timedelta, timezone
from decimal import Decimal
import pandas as pd
import pymysql
from pymysqlreplication import BinLogStreamReader
from pymysqlreplication.row_event import WriteRowsEvent, UpdateRowsEvent, DeleteRowsEvent
from pymysqlreplication.event import XidEvent

MYSQL = dict(host=os.getenv("MYSQL_HOST", "127.0.0.1"),
             port=int(os.getenv("MYSQL_PORT", "3306")),
             user=os.getenv("MYSQL_USER", "root"), password=os.getenv("MYSQL_PASSWORD", "root"),
             charset="utf8mb4", connect_timeout=10, read_timeout=30)

def sql(statement, params=None):
    with pymysql.connect(**MYSQL, autocommit=True, cursorclass=pymysql.cursors.DictCursor) as conn:
        with conn.cursor() as cur:
            cur.execute("SET time_zone = '+00:00'")
            cur.execute(statement, params)
            return list(cur.fetchall()) if cur.description else cur.rowcount

def read_settings():
    return sql("SELECT @@version AS version, @@server_uuid AS uuid, @@server_id AS server_id, "
               "@@log_bin AS log_bin, @@binlog_format AS format, @@binlog_row_image AS image, "
               "@@binlog_row_metadata AS metadata, @@binlog_transaction_compression AS compressed")[0]

settings = read_settings()
print("Current server settings:", settings)

## Enable CDC: server binary logging, then table filtering

MySQL does **not** have an `ENABLE CDC` command for an individual database or table.
Binary logging is enabled on the **server**. Later, our Python reader selects only
`DB` and its `products` table with `only_schemas=[DB]` and `only_tables=["products"]`.
`order_events` is exported separately using its increasing event ID. Other tables may
still be present in the server binlog; reader filters are not a security boundary.

### Ubuntu 24.04 / MySQL 8.x: edit the server configuration

Run these commands in an **Ubuntu terminal on the MySQL server**, not in a Python cell.
Check the installed server version and current service status:

```bash
mysqld --version
sudo systemctl status mysql --no-pager
```

If binary logging is disabled or startup settings need changing, stop MySQL and open
the Ubuntu package's server configuration with nano. Stopping MySQL disconnects clients.

```bash
sudo systemctl stop mysql
sudo nano /etc/mysql/mysql.conf.d/mysqld.cnf
```

Merge these options into the existing `[mysqld]` section; update existing entries instead
of adding conflicting duplicates. Keep the other configuration settings intact.
Use a nonzero `server-id` unique in your replication environment; 47 is only a lab example.
Remove any conflicting `skip-log-bin` or `disable-log-bin` startup option.

```ini
[mysqld]
server-id=47
log-bin=mysql-bin
binlog-format=ROW
binlog-row-image=FULL
binlog-row-metadata=FULL
binlog-transaction-compression=OFF
```

In nano, press **Ctrl+O**, **Enter** to save, then **Ctrl+X** to exit.
Restart MySQL and confirm that the service reports `active (running)`:

```bash
sudo systemctl restart mysql
sudo systemctl status mysql --no-pager
```

`restart` also starts a stopped service. To start it without a restart, use
`sudo systemctl start mysql`. If startup fails, inspect the service log:

```bash
sudo journalctl -u mysql -n 50 --no-pager
```

Rerun the notebook's connection cell after the service is running. Its `SELECT @@version`
query checks the connected server version. These instructions target MySQL 8.0/8.4;
the binlog-position helper below supports both versions' status commands.
`log_bin` is a startup setting: SQL alone cannot enable it on an already running server.
If binary logging is already enabled and the server ID is valid, skip the stop/edit/restart
steps and continue with the verification cell below.
If server binlog include/exclude filters already exist, ensure they do not exclude the
lab database. Keep binlogs long enough to capture all changes.

### Verify the server settings

Assume CDC has already been configured at the server level. Run the next cell **before
Change 0** to read and verify the required settings. It does not change server variables
or persistent configuration. If verification fails, the message lists each incorrect
setting and its expected value. Have the server configuration corrected outside this
notebook, then rerun the connection and verification cells. The Ubuntu terminal commands
above are a reference for that separate administrator setup.

References: [Ubuntu MySQL configuration](https://ubuntu.com/server/docs/how-to/databases/install-mysql/),
[MySQL binary logging options](https://dev.mysql.com/doc/refman/8.0/en/replication-options-binary-log.html).

In [ ]:
settings = read_settings()
required = {
    "log_bin": ("log_bin", 1),
    "format": ("binlog_format", "ROW"),
    "image": ("binlog_row_image", "FULL"),
    "metadata": ("binlog_row_metadata", "FULL"),
    "compressed": ("binlog_transaction_compression", 0),
}
problems = []
for key, (variable, expected) in required.items():
    if settings[key] != expected:
        problems.append(f"{variable}: found {settings[key]!r}; required {expected!r}")
if settings["server_id"] <= 0:
    problems.append(f"server_id: found {settings['server_id']}; required a nonzero, unique server ID")
if problems:
    raise RuntimeError("CDC verification failed: " + "; ".join(problems) +
        ". Correct the MySQL server configuration outside this notebook, restart MySQL if needed, "
        "then rerun connection and verification. No server settings were changed by this cell.")
READER_ID = int(os.getenv("CDC_READER_ID", "470047"))
if READER_ID <= 0 or READER_ID == settings["server_id"]:
    raise ValueError("Choose a positive CDC_READER_ID different from the MySQL server_id.")
print("CDC ready:", settings)
print("Capture scope:", DB, "table: products (created in the next cell)")

## Create the source tables and save the initial binlog position

`order_id` is the **business key**: one order has multiple lifecycle events.
`event_id` identifies one physical event row. `created_at` records arrival; `event_time`
records business time. They need not have the same ordering.

We save the binlog offset **before inserting initial data**. This avoids needing an initial
snapshot. Do not remove local checkpoints while retaining the source database. A new
`DB` gives a clean demonstration without deleting the previous one.

In [ ]:
sql(f"CREATE DATABASE IF NOT EXISTS `{DB}`")
sql(f"""CREATE TABLE IF NOT EXISTS `{DB}`.products (
    product_id BIGINT PRIMARY KEY, product_name VARCHAR(100) NOT NULL,
    category VARCHAR(50), color VARCHAR(30), list_price DECIMAL(10,2),
    updated_at DATETIME(6) NOT NULL) ENGINE=InnoDB""")
sql(f"""CREATE TABLE IF NOT EXISTS `{DB}`.order_events (
    event_id BIGINT AUTO_INCREMENT PRIMARY KEY, order_id BIGINT NOT NULL,
    product_id BIGINT NOT NULL, quantity INT NOT NULL, selling_price DECIMAL(10,2) NOT NULL,
    status VARCHAR(20) NOT NULL, event_time DATETIME(6) NOT NULL,
    created_at DATETIME(6) NOT NULL) ENGINE=InnoDB""")
sql(f"""CREATE TABLE IF NOT EXISTS `{DB}`.applied_changes (
    change_id INT PRIMARY KEY, applied_at DATETIME(6) NOT NULL) ENGINE=InnoDB""")

def binlog_position():
    try:
        return sql("SHOW BINARY LOG STATUS")[0]  # MySQL 8.4+
    except pymysql.err.ProgrammingError as exc:
        if exc.args[0] != 1064:
            raise
        return sql("SHOW MASTER STATUS")[0]  # MySQL 8.0

START = LAB_DIR / "start.json"
BATCHES = LAB_DIR / "batches"
BATCHES.mkdir(exist_ok=True)
if not START.exists():
    assert sql(f"SELECT COUNT(*) AS n FROM `{DB}`.applied_changes")[0]["n"] == 0, "Use a new DB."
    assert sql(f"SELECT COUNT(*) AS n FROM `{DB}`.products")[0]["n"] == 0, "Source must start empty."
    assert sql(f"SELECT COUNT(*) AS n FROM `{DB}`.order_events")[0]["n"] == 0
    pos = binlog_position()
    START.write_text(json.dumps(dict(log_file=pos["File"], log_pos=pos["Position"],
        order_id=0, batch_id=0, sequence=0, last_time="1970-01-01 00:00:00.000000",
        source_uuid=settings["uuid"], database=DB)))
print("Saved starting position:", json.loads(START.read_text()))

## Small helper: commit each named change once

The change marker and business rows commit in the same transaction. Repeating a change is
a no-op. Changes must be applied in order. SQL for each actual change remains visible below.

In [ ]:
def apply_change(number, product_sql, products_params, events):
    with pymysql.connect(**MYSQL) as conn:
        with conn.cursor() as cur:
            cur.execute("SET time_zone = '+00:00'")
            cur.execute(f"SELECT change_id FROM `{DB}`.applied_changes ORDER BY change_id")
            done = [r[0] for r in cur.fetchall()]
            if number in done:
                print("Already applied change", number)
                return
            assert done == list(range(number)), "Run preceding changes first."
            now = datetime.now(timezone.utc).replace(tzinfo=None)
            for statement, params in zip(product_sql, products_params):
                cur.execute(statement.format(db=DB), (*params, now))
            for order_id, product_id, qty, price, status, minutes_ago in events:
                cur.execute(f"""INSERT INTO `{DB}`.order_events
                    (order_id, product_id, quantity, selling_price, status, event_time, created_at)
                    VALUES (%s,%s,%s,%s,%s,%s,%s)""",
                    (order_id, product_id, qty, price, status, now-timedelta(minutes=minutes_ago), now))
            cur.execute(f"INSERT INTO `{DB}`.applied_changes VALUES (%s,%s)", (number, now))
        conn.commit()
    display(pd.DataFrame(sql(f"SELECT * FROM `{DB}`.products ORDER BY product_id")))
    display(pd.DataFrame(sql(f"SELECT * FROM `{DB}`.order_events ORDER BY event_id")))

## Small CDC collector: committed binlog rows → immutable batch

Only **one writer and one collector** operate this lab. Stop source writes while capturing;
we verify the binlog did not move during capture. This deliberately avoids a production
snapshot/concurrency framework. Retain MySQL binlogs until captured; a purged checkpoint
requires a new lab run, not silently skipping missing history.

Rows are buffered until `XidEvent` (transaction commit). INSERT/UPDATE use after values;
DELETE uses before values. UPDATE retains both images. `cdc_id` comes from the binlog
file, position and row index; `sequence` preserves source order, including multiple changes
in one batch. Binlog timestamps have second precision. `event_time` preserves those times
with a monotonic microsecond tie-break for equal timestamps; original `binlog_time` and
`source_updated_at` are retained. This gives nonempty, deterministic SCD validity intervals.

A completed directory contains JSON, incremental order CSV and one manifest/checkpoint.
An atomic directory rename publishes the batch; previous batches are never overwritten.
An interrupted `.pending_*` directory is ignored. This is a small collector.

In [ ]:
def json_value(value):
    if isinstance(value, (datetime, Decimal)):
        return str(value)
    raise TypeError(type(value).__name__)

def capture():
    manifests = sorted(BATCHES.glob("batch_*/manifest.json"))
    state = json.loads((manifests[-1] if manifests else START).read_text())
    assert state["source_uuid"] == settings["uuid"] and state["database"] == DB
    end = binlog_position()
    batch_id = state["batch_id"] + 1
    changes, pending = [], []
    stream = BinLogStreamReader(connection_settings=dict(MYSQL), server_id=READER_ID,
        log_file=state["log_file"], log_pos=state["log_pos"], resume_stream=True,
        blocking=False, only_schemas=[DB], only_tables=["products"],
        only_events=[WriteRowsEvent, UpdateRowsEvent, DeleteRowsEvent, XidEvent],
        enable_logging=False)
    last_time = datetime.fromisoformat(state["last_time"])
    sequence = state["sequence"]
    try:
        for event in stream:
            if isinstance(event, XidEvent):
                changes.extend(pending)
                pending = []
                continue
            for index, row in enumerate(event.rows):
                operation = "I" if isinstance(event, WriteRowsEvent) else "U" if isinstance(event, UpdateRowsEvent) else "D"
                before = row.get("before_values", row.get("values") if operation == "D" else None)
                after = row.get("after_values", row.get("values") if operation == "I" else None)
                values = after if after is not None else before
                assert "product_id" in values, "FULL row metadata required before source writes."
                binlog_time = datetime.fromtimestamp(event.timestamp, timezone.utc).replace(tzinfo=None)
                last_time = max(binlog_time, last_time + timedelta(microseconds=1))
                sequence += 1
                pending.append(dict(batch_id=batch_id, sequence=sequence,
                    cdc_id=f"{stream.log_file}:{event.packet.log_pos}:{index}", operation=operation,
                    **{k: values[k] for k in ("product_id", "product_name", "category", "color", "list_price")},
                    event_time=str(last_time), binlog_time=str(binlog_time),
                    source_updated_at=str(values["updated_at"]), before=before, after=after))
    finally:
        stream.close()
    assert not pending, "Uncommitted rows: stop writers and retry."
    orders = sql(f"SELECT * FROM `{DB}`.order_events WHERE event_id > %s ORDER BY event_id", (state["order_id"],))
    end_check = binlog_position()
    assert (end["File"], end["Position"]) == (end_check["File"], end_check["Position"]), "Source moved; stop writers and retry."
    if not changes and not orders:
        print("No new source rows. No batch created.")
        return
    import uuid
    staging = BATCHES / (".pending_" + uuid.uuid4().hex)
    staging.mkdir()
    (staging / f"product_cdc_{batch_id:03d}.json").write_text(
        "".join(json.dumps(r, default=json_value) + "\n" for r in changes))
    fields = ["event_id", "order_id", "product_id", "quantity", "selling_price", "status", "event_time", "created_at"]
    with (staging / f"orders_{batch_id:03d}.csv").open("w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        writer.writeheader()
        writer.writerows(orders)
    next_state = dict(log_file=end["File"], log_pos=end["Position"], batch_id=batch_id,
        order_id=max([state["order_id"]] + [r["event_id"] for r in orders]), sequence=sequence,
        last_time=str(last_time), source_uuid=settings["uuid"], database=DB,
        product_rows=len(changes), order_rows=len(orders))
    (staging / "manifest.json").write_text(json.dumps(next_state, indent=2))
    staging.rename(BATCHES / f"batch_{batch_id:06d}")
    print(f"Published batch {batch_id}: {len(changes)} product changes, {len(orders)} order events")
    display(pd.DataFrame(changes).drop(columns=["before", "after"], errors="ignore"))

## Change 0 — Initial products and orders at 100

Run the change cell, then its capture cell. Run notebooks 2 and 3 to inspect the resulting state.

In [ ]:
apply_change(0,
    ["INSERT INTO `{db}`.products VALUES (1,%s,%s,%s,%s,%s)",
     "INSERT INTO `{db}`.products VALUES (2,%s,%s,%s,%s,%s)",
     "INSERT INTO `{db}`.products VALUES (3,%s,%s,%s,%s,%s)"],
    [("Training Laptop", "Premium", "Black", "100.00"),
     ("Training Mouse", "Accessories", "Black", "20.00"),
     ("Training Keyboard", "Accessories", "White", "40.00")],
    [(1001,1,1,"100.00","CREATED",60), (1001,1,1,"100.00","PAID",50),
     (1002,1,2,"100.00","CREATED",60), (1002,1,2,"100.00","PAID",45),
     (1003,1,1,"100.00","CREATED",40)])

In [ ]:
capture()

## Change 1 — Price 100 → 95; orders reach different states

Run the change cell, then its capture cell. Run notebooks 2 and 3 to inspect the resulting state.

In [ ]:
apply_change(1, ["UPDATE `{db}`.products SET list_price=%s, updated_at=%s WHERE product_id=1"],
    [("95.00",)],
    [(1001,1,1,"100.00","DELIVERED",0), (1002,1,2,"100.00","CANCELLED",0),
     (1003,1,1,"100.00","PAID",0), (1004,1,2,"95.00","CREATED",0),
     (1005,1,1,"95.00","PAID",0)])

In [ ]:
capture()

## Change 2 — Color Black → Silver

Run the change cell, then its capture cell. Run notebooks 2 and 3 to inspect the resulting state.

In [ ]:
apply_change(2, ["UPDATE `{db}`.products SET color=%s, updated_at=%s WHERE product_id=1"],
    [("Silver",)], [(1006,1,1,"95.00","CREATED",0), (1006,1,1,"95.00","PAID",0)])
# Both 1006 rows share event_time; event_id deterministically picks PAID.

In [ ]:
capture()

## Change 3 — Category Premium → Standard; an older event arrives late

Run the change cell, then its capture cell. Run notebooks 2 and 3 to inspect the resulting state.

In [ ]:
apply_change(3, ["UPDATE `{db}`.products SET category=%s, updated_at=%s WHERE product_id=1"],
    [("Standard",)], [(1007,1,3,"95.00","PAID",0)])
# Add late PAID with business time before the existing DELIVERED event.
# This is a separate idempotent INSERT, so retrying the cell remains safe.
sql(f"""INSERT INTO `{DB}`.order_events
    (order_id,product_id,quantity,selling_price,status,event_time,created_at)
    SELECT 1001,1,1,100.00,'PAID',DATE_SUB(event_time, INTERVAL 1 MINUTE),UTC_TIMESTAMP(6)
    FROM `{DB}`.order_events d WHERE d.order_id=1001 AND d.status='DELIVERED'
    AND NOT EXISTS (SELECT 1 FROM `{DB}`.order_events p WHERE p.order_id=1001
                    AND p.status='PAID' AND p.event_time=DATE_SUB(d.event_time, INTERVAL 1 MINUTE))""")
display(pd.DataFrame(sql(f"SELECT * FROM `{DB}`.order_events WHERE order_id=1001 ORDER BY event_id")))

In [ ]:
capture()

## Change 4 — Price 95 → 90

Run the change cell, then its capture cell. Run notebooks 2 and 3 to inspect the resulting state.

In [ ]:
apply_change(4, ["UPDATE `{db}`.products SET list_price=%s, updated_at=%s WHERE product_id=1"],
    [("90.00",)], [(1008,1,2,"90.00","PAID",0), (1009,1,1,"90.00","CREATED",0)])

In [ ]:
capture()

## Change 5 — Unchanged tracked attributes (no extra SCD2 version)

Optional edge-case check. Run the change cell, then its capture cell. Run notebooks 2 and 3 to inspect the resulting state.

In [ ]:
apply_change(5, ["UPDATE `{db}`.products SET updated_at=%s WHERE product_id=1"], [()], [])
# MySQL emits an UPDATE, but only updated_at changes. Category/color/price/name are identical.

In [ ]:
capture()

## Change 6 — Delete product 3 (expire history; remove current state)

Optional edge-case check. Run the change cell, then its capture cell. Run notebooks 2 and 3 to inspect the resulting state.

In [ ]:
apply_change(6, ["DELETE FROM `{db}`.products WHERE product_id=3 AND updated_at <= %s"], [()], [])

In [ ]:
capture()

## Change 7 — Reinsert product 3 (a new lifetime, even with identical attributes)

Optional edge-case check. Run the change cell, then its capture cell. Run notebooks 2 and 3 to inspect the resulting state.

In [ ]:
apply_change(7, ["INSERT INTO `{db}`.products VALUES (3,%s,%s,%s,%s,%s)"],
    [("Training Keyboard", "Accessories", "White", "40.00")], [])

In [ ]:
capture()

## Inspect all published batches

Rerun `capture()` whenever you need to collect new committed changes. An unchanged source
produces no extra batch. Leave existing batches immutable for downstream replay.

In [ ]:
display(pd.DataFrame([json.loads(p.read_text()) for p in sorted(BATCHES.glob("batch_*/manifest.json"))]))